<!-- nav -->
[← 3 · Which standard?](03_Which_Standard.ipynb) · [Home](../../README.md) · [5 · One framework, three languages →](05_One_Framework_Three_Languages.ipynb)

# 4 · What names refer to

**The problem.** Two functions each have a local `x`. Renaming one of them by finding every `Identifier` spelled `x`
renames the other too, and an attribute `point.x` that has nothing to do with either. A name means something only in
its scope.

`define(module)` reads a module the way Python does: which **entities** it binds, in which **scopes**, and what each
name refers to.

In [ ]:
from mbse.Programs.Framework.Syntax import walk
from mbse.Programs.Python import Definitions, Python314, Syntax as P

source = '''def area(w, h):
    x = w * h
    return x


def perimeter(w, h):
    x = 2 * (w + h)
    return x


point.x = area(2, 3)
'''

## Step 1: the naive rename

Renaming every `Identifier` spelled `x` changes three things that aren't one:

In [ ]:
naive = Python314.parse(source)
for node in walk(naive):
    if isinstance(node, P.Identifier) and node.spelling == "x":
        node.spelling = "result"
print(Python314.print(naive))

## Step 2: entities and scopes

`Definitions.define` gives the module's **program**: the entities, each in its scope, with a qualified name:

In [ ]:
module = Python314.parse(source)
program = Definitions.define(module)
list(program.entities())

There are two variables called `x`, `area.x` and `perimeter.x`, and none for `point.x`: an attribute's name depends
on the object, which isn't known until the program runs.

## Step 3: lookup and referents

`program.lookup(name, at)` resolves a name from a place in the tree, as Python would there. `referents` gives what a
`Name` in the tree refers to:

In [ ]:
area = module.body[0]
[x] = program.lookup("x", area.body[0])
print(x, x.declarations)
names = [node for node in walk(module) if isinstance(node, P.Name) and node.id.spelling == "x"]
[(Python314.print(name).strip(), Definitions.referents(program, name)) for name in names]

## Step 4: the safe rename

Rename only the names that refer to `area.x`:

In [ ]:
for name in names:
    if x in Definitions.referents(program, name):
        name.id.spelling = "result"
print(Python314.print(module))

## Step 5: Python's own rules

Python's scoping has corners, and `define` follows them: a name assigned in a function is local to it unless `global`
or `nonlocal` says otherwise, and a method doesn't see its class body's names.

In [ ]:
tricky = Python314.parse('''count = 0
LIMIT = 3


class Counter:
    LIMIT = 10

    def bump(self):
        global count
        count += 1
        return count < LIMIT
''')
program = Definitions.define(tricky)
bump = tricky.body[2].body[1]
for name in [node for node in walk(bump) if isinstance(node, P.Name) and node.id.spelling in ("count", "LIMIT")]:
    print(name.id.spelling, Definitions.referents(program, name))

`count` is the module's, because of `global`; `LIMIT` is the module's too, not `Counter.LIMIT`, because lookup from a
method skips the class body.

## Why it works this way

- **Names are resolved, not matched.** A rename, an inline, a dead-code check all need to know what a name refers to.
  Spelling is only the start.
- **Each language's own rules.** Python's scopes are Python's; C++'s namespaces and TypeScript's values and types are
  theirs (case study 5). Each language's `Definitions` follows its standard.
- **What can't be known statically isn't guessed.** Attributes, keyword arguments and imported modules' members depend
  on values, so they find nothing rather than something wrong.

<!-- nav -->
[← 3 · Which standard?](03_Which_Standard.ipynb) · [Home](../../README.md) · [5 · One framework, three languages →](05_One_Framework_Three_Languages.ipynb)